# CS781 Capstone - Phase 7: Prior Case Retrieval
## Rescoring notebook - **post-processing and ensembling of saved runs** (no GPU, no retraining)

**What this does.** Takes the score files that earlier runs saved (`scores_dev.npz`, `scores_test.npz`, and `*_embeddings.npz` for PRF) and re-ranks with the dev-tuned post-processing: weighted fusion of the runs with BM25 / BM25-on-short-queries / statute overlap / a temporal prior, embedding-space PRF, cross-query competition, and **refinement rounds** (competition and embedding feedback alternate: the competition makes each query's top set cleaner, the centroid of that cleaner set is a better query for the next round). **No GPU, no model, no training** - it runs in a few minutes on CPU.

**New: the fusion space is chosen on held-out dev queries.** The dev split is small, and tuning a separate weight for every pooled / q2c / c2q score of every run fits its noise (an earlier run: dev F1 0.43, grader 0.39). The notebook now compares three spaces by repeated cross-validation - `ensemble` (mean of the runs, only lexical weights fitted), `runs` (one weight per run) and `split` (every component separately) - and uses the simplest one within `SPACE_MARGIN` of the best. Force one with `FUSION_SPACE`.

With one run attached it is a pure post-processing upgrade of that run; with several (e.g. exp 2, 3 and 4) it also ensembles them. The competition step is **transductive** (it compares each candidate across all queries of the split, no labels); it is dev-tuned, only stays on if it gains, and `no_competition/prediction.json` (and, when the refinement rounds are on, `no_refinement/prediction.json`) is written next to the final file so you can check each step's real value on the grader. Set `USE_REFINE=False` / `USE_COMPETE=False` for a less transductive system.

**Attach** the output folders of the earlier runs (`cs781_phase7_<exp>`) as Kaggle inputs (or put them under `/kaggle/working`); they are found automatically. Only folders that hold both `scores_dev.npz` and `scores_test.npz` and whose queries/candidates match the data loaded here are used. The Phase-7 data comes from the HF dataset repo via the Kaggle secret `HF`.

**Rules respected.** No new model, no external model or API; classical statistics only; each split uses only its own pool; a query never retrieves itself.

### 1. Setup and configuration
All knobs are in `CFG`. `P7_CFG_OVERRIDE` (JSON in an env var) overrides them without editing the cell.

In [ ]:
# =============================================================================
# CELL 1 - setup, configuration, small utilities
# CS781 Phase 7 (PCR) - Rescoring of saved runs (no GPU): fusion, PRF and cross-query competition
# =============================================================================
import os, sys, json, re, math, time, csv, random, zipfile, shutil, hashlib, warnings, gc, types, subprocess
from dataclasses import dataclass, asdict, fields
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np


def _ensure(pip_name, import_name):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


_ensure("tokenizers", "tokenizers")
_ensure("huggingface_hub", "huggingface_hub")
_ensure("scikit-learn", "sklearn")
_ensure("scipy", "scipy")
_ensure("pandas", "pandas")

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# CONFIG.  Everything you may want to change is here.
# Quick smoke run on Kaggle without editing the cell (set BEFORE running this cell):
#   os.environ["P7_CFG_OVERRIDE"] = '{"EPOCHS":1,"MAX_QUERIES":40,"MAX_POOL":400,"SSL_EPOCHS":1,"REFRESH_NEGS_AT":[]}'
# -----------------------------------------------------------------------------
CFG = dict(
    # ---- identity
    EXP_NAME='rescore_saved_runs',
    VARIANT='lormaplus',
    # ---- where things are (None = auto-discover under /kaggle/input)
    P7_ROOT=None,
    HF_DATASET_REPO='Exploration-Lab/CS-781-Capstone',
    # ---- long-document handling
    STRIP_PLACEHOLDERS=True,
    MAX_CHARS=100000,
    # ---- document representation
    TOKEN_POOL='mean',
    DOC_MODE='pooled',
    DOC_POOL='mean',
    # ---- lexical signals + fusion (weights are tuned on dev)
    USE_BM25=True,
    USE_REFS=True,
    USE_PRF=True,
    BM25_NGRAM=(1, 1),
    BM25_K1=1.2,
    BM25_B=0.75,
    MIN_GAIN=0.002,
    USE_BM25_TOP=True,
    BM25_TOPS=(30, 60, 120, 250, 500),
    USE_TIME=True,
    USE_COMPETE=True,
    USE_REFINE=True,
    USE_CITE=True,
    USE_TPRIOR=True,
    COMPETE_MIN_GAIN=0.01,
    CV_FOLDS=5,
    # ---- evaluation / output
    K=7,
    SEED=1234,
    MAX_QUERIES=None,
    MAX_POOL=None,
    # ---- rescoring notebook only (None = auto-discover score files under /kaggle/input)
    RUN_DIRS=None,
    RUN_EXCLUDE=(),
    FUSION_SPACE='auto',
    LEX_MODE='auto',
    CV_REPEATS=3,
    SPACE_MARGIN=0.005,
)
_ov = os.environ.get("P7_CFG_OVERRIDE")
if _ov:
    CFG.update(json.loads(_ov))
for _k in ("LORMA_TARGETS", "BM25_NGRAM", "BM25_TOPS", "REFRESH_NEGS_AT"):          # JSON gives lists; not every variant has every key
    if _k in CFG:
        CFG[_k] = tuple(CFG[_k])
C = types.SimpleNamespace(**CFG)
assert C.TOKEN_POOL in ("mean", "last") and C.DOC_POOL in ("mean", "max", "first", "attn")
assert C.DOC_MODE in ("pooled", "hier") and C.VARIANT in ("lormaplus", "partial", "full")

# -----------------------------------------------------------------------------
# paths / device
# -----------------------------------------------------------------------------
IS_KAGGLE = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
BASE_DIR = "/kaggle/working" if IS_KAGGLE else "."
WORK_DIR = os.path.join(BASE_DIR, "p7_" + C.EXP_NAME)
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
OUT_DIR = os.path.join(BASE_DIR, "cs781_phase7_" + C.EXP_NAME)       # prediction.json + submission.zip land here
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = {"enabled": False, "dtype": torch.float16}
if DEVICE.type == "cuda":
    AMP["enabled"] = True
    AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m{sec % 60:02d}s"


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def walk_find(filename, roots):
    """all directories under `roots` that contain a file called `filename`"""
    hits, seen = [], set()
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files and d not in seen:
                seen.add(d)
                hits.append(d)
    return hits


def read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def hf_token():
    # Kaggle secret named HF, or env HF_TOKEN / HF.  Nothing is hard-coded in this notebook.
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HF") or None


HF_TOKEN = hf_token()
set_seed(C.SEED)
RESULTS = {"config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}}
T_START = time.time()

log(f"torch {torch.__version__} | device {DEVICE} | "
    f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'no GPU (none is needed here)'} | "
    f"autocast {AMP['dtype'] if AMP['enabled'] else 'off'}")
log(f"experiment {C.EXP_NAME} | variant {C.VARIANT} | HF token available: {bool(HF_TOKEN)} | output dir {OUT_DIR}")

### 3. Phase-7 data
Queries and candidates per split; each split uses only its own pool. Anonymisation tags (`<NAME>`, `<ORG>`, `<CITATION>`) are stripped.

In [ ]:
# =============================================================================
# CELL 3 - Phase-7 data: load, normalise, per-split structures
# Each split uses ONLY its own candidate pool (train<->train_candidates, dev<->dev_candidates, test<->test_candidates).
# =============================================================================
P7_FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}


def _looks_like_p7(path):
    p = path.lower().replace("-", "_")
    return "phase_7" in p or "phase7" in p or "pcr" in p


def p7_file(rel):
    """local copy (P7_ROOT / Kaggle input) first, then the Hugging Face dataset repo"""
    base = os.path.basename(rel)
    if C.P7_ROOT:
        for cand in (Path(C.P7_ROOT) / rel, Path(C.P7_ROOT) / base):
            if cand.is_file():
                return str(cand)
        for d in walk_find(base, [C.P7_ROOT]):
            return os.path.join(d, base)
    for d in walk_find(base, [INPUT_ROOT]):                         # an attached copy of the dataset
        if _looks_like_p7(d):
            return os.path.join(d, base)
    err = None
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=C.HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        err = e
    raise FileNotFoundError(f"cannot find {rel}: not under P7_ROOT / {INPUT_ROOT} and the HF download failed: {err!r}")


def unwrap_json(obj):
    if isinstance(obj, dict):
        for key in ("data", "records", "examples", "queries", "candidates"):
            if isinstance(obj.get(key), (list, dict)):
                return obj[key]
    return obj


def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):                                     # sentence list -> one string, order preserved
        return " ".join(str(x).strip() for x in value if x is not None).strip()
    return str(value).strip()


def records_from_queries(obj):
    """-> list of (query_id, text, [relevant ids])"""
    rows = []
    if isinstance(obj, list):
        iterator = enumerate(obj)
    elif isinstance(obj, dict):
        iterator = [(None, obj)] if any(k in obj for k in ("id", "case_id", "query_id")) else obj.items()
    else:
        raise TypeError(f"queries must be list/dict, got {type(obj)}")
    for key, rec in iterator:
        if not isinstance(rec, dict):
            continue
        cid = rec.get("id", rec.get("case_id", rec.get("query_id", key)))
        text = rec.get("text", rec.get("document", rec.get("case_text", "")))
        rel = rec.get("relevant_candidates", rec.get("relevant_candidate_ids", [])) or []
        if isinstance(rel, str):
            rel = [rel]
        rows.append((str(cid), normalize_text(text), [str(x) for x in rel]))
    return rows


def records_from_candidates(obj):
    """-> dict candidate_id -> text"""
    out = {}
    if isinstance(obj, dict):
        iterable = [(None, obj)] if any(k in obj for k in ("id", "case_id", "text", "document")) else obj.items()
    elif isinstance(obj, list):
        iterable = [(None, r) for r in obj]
    else:
        raise TypeError(f"candidates must be list/dict, got {type(obj)}")
    for key, rec in iterable:
        if isinstance(rec, dict):
            cid = rec.get("id", rec.get("case_id", rec.get("candidate_id", key)))
            txt = rec.get("text", rec.get("document", rec.get("case_text", "")))
        else:
            cid, txt = key, rec
        if cid is None:
            raise ValueError("cannot infer a candidate id - inspect the raw candidate file")
        out[str(cid)] = normalize_text(txt)
    return out


_PLACEHOLDER = re.compile(r"<\s*[A-Z][A-Z_]{1,19}\s*>")             # <NAME> <ORG> <CITATION> ...


def prep_text(t):
    if C.STRIP_PLACEHOLDERS:
        t = _PLACEHOLDER.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t[:C.MAX_CHARS]


def limit_split(sp, max_q, max_pool, seed):
    """smoke-test helper: first `max_q` queries; pool = their gold + their own ids + random fill up to `max_pool`"""
    rng = random.Random(seed)
    qids = sp.qids[:max_q] if max_q else list(sp.qids)
    pool = set(sp.cids)
    keep = set()
    for q in qids:
        keep |= (sp.gold[q] & pool)
        if q in pool:
            keep.add(q)
    rest = [c for c in sp.cids if c not in keep]
    rng.shuffle(rest)
    n_fill = len(rest) if not max_pool else max(max_pool - len(keep), 0)
    keep |= set(rest[:n_fill])
    sp.qids = qids
    sp.cids = [c for c in sp.cids if c in keep]
    sp.qtext = {q: sp.qtext[q] for q in qids}
    sp.gold = {q: sp.gold[q] for q in qids}
    sp.ctext = {c: sp.ctext[c] for c in sp.cids}
    return sp


def load_split(name):
    q_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_queries"]), encoding="utf-8")))
    c_raw = unwrap_json(json.load(open(p7_file(P7_FILES[f"{name}_candidates"]), encoding="utf-8")))
    qrows = records_from_queries(q_raw)
    cmap = records_from_candidates(c_raw)
    sp = types.SimpleNamespace(name=name)
    sp.qids = [q for q, _, _ in qrows]
    sp.qtext = {q: prep_text(t) for q, t, _ in qrows}
    sp.gold = {q: set(r) for q, _, r in qrows}
    sp.cids = list(cmap.keys())
    sp.ctext = {c: prep_text(t) for c, t in cmap.items()}
    if C.MAX_QUERIES or C.MAX_POOL:
        sp = limit_split(sp, C.MAX_QUERIES, C.MAX_POOL, C.SEED)
        log(f"[smoke] {name}: limited to {len(sp.qids)} queries / {len(sp.cids)} candidates")
    pool = set(sp.cids)
    sp.gold_in_pool = {q: sorted((sp.gold[q] & pool) - {q}) for q in sp.qids}
    return sp


SP = {name: load_split(name) for name in ("train", "dev", "test")}

rows = []
for name, sp in SP.items():
    pool = set(sp.cids)
    ng = [len(sp.gold[q]) for q in sp.qids]
    nl = [len(sp.qtext[q].split()) for q in sp.qids]
    cl = [len(t.split()) for t in sp.ctext.values()]
    mism = sum(1 for q in sp.qids if q in sp.ctext and sp.ctext[q] != sp.qtext[q])
    rows.append({
        "split": name, "queries": len(sp.qids), "pool": len(sp.cids),
        "query_in_pool": sum(q in pool for q in sp.qids), "text_differs": mism,
        "gold_mean": round(float(np.mean(ng)), 2) if ng else 0, "gold_min": min(ng) if ng else 0,
        "gold_max": max(ng) if ng else 0,
        "gold_not_in_pool": sum(len(sp.gold[q] - pool) for q in sp.qids),
        "no_gold_queries": sum(len(sp.gold[q]) == 0 for q in sp.qids),
        "q_words_med": int(np.median(nl)) if nl else 0, "c_words_med": int(np.median(cl)) if cl else 0,
        "c_words_p95": int(np.percentile(cl, 95)) if cl else 0,
    })
log(pd.DataFrame(rows).to_string(index=False))
log("(query_in_pool > 0 means a query's own case is also in the pool: it is excluded from its own ranking)")
assert all(len(SP[s].qids) > 0 and len(SP[s].cids) > C.K for s in SP), "a split is empty / its pool is smaller than K+1"
assert sum(len(SP["train"].gold_in_pool[q]) > 0 for q in SP["train"].qids) > 0, "no train query has a gold candidate in the pool"
assert any(len(SP["dev"].gold[q]) > 0 for q in SP["dev"].qids), "dev has no labels"
RESULTS["data"] = rows
sample_txt = next(iter(SP["train"].qtext.values()))
log("sample (first 300 chars of a prepared train query):", sample_txt[:300])

### 5. Retrieval utilities
Ranking, micro-P/R/F1@7, BM25, legal-reference overlap, dev-tuned score fusion, pseudo-relevance feedback, hard-negative mining (numpy / scikit-learn, with self-tests).

In [ ]:
# =============================================================================
# CELL 5 - retrieval utilities (numpy / scipy / scikit-learn only):
#   ranking, micro-P/R/F1@K, BM25, legal-reference overlap, score fusion tuned on dev, pseudo-relevance feedback,
#   hard-negative mining.  Everything here has a small self-test.
# =============================================================================
def l2n(x, eps=1e-8):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)


def zrows(S):
    """per-query z-score of a score matrix [n_queries, n_candidates]"""
    S = S.astype(np.float32, copy=False)
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)


def mask_self(S, qids, cids):
    """-inf where a candidate is the query itself (queries also sit in their pool). Works on a copy."""
    S = S.astype(np.float32, copy=True)
    cidx = {c: i for i, c in enumerate(cids)}
    for i, q in enumerate(qids):
        j = cidx.get(q)
        if j is not None:
            S[i, j] = -np.inf
    return S


def rank_topk(S, qids, cids, k):
    """top-k candidate ids per query (best first), the query's own id excluded.  S: [n_queries, n_candidates]"""
    S = mask_self(S, qids, cids)
    k = min(k, S.shape[1] - 1)
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    sc = np.take_along_axis(S, part, axis=1)
    order = np.argsort(-sc, axis=1, kind="stable")
    top = np.take_along_axis(part, order, axis=1)
    return {q: [cids[j] for j in top[i]] for i, q in enumerate(qids)}


def micro_prf(preds, gold, qids, k=7):
    """micro P / R / F1@k against the FULL gold set (no truncation, no padding)"""
    tp = n_pred = n_gold = 0
    for q in qids:
        p = list(preds[q][:k])
        g = gold[q]
        tp += len(set(p) & g)
        n_pred += len(p)
        n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {"P": P, "R": R, "F1": F1, "tp": tp, "n_pred": n_pred, "n_gold": n_gold}


def f1_of(S, sp, k):
    return micro_prf(rank_topk(S, sp.qids, sp.cids, k), sp.gold, sp.qids, k)["F1"]


def first_hit_rank(S, qids, cids, gold):
    """1-based rank of the best-ranked gold candidate (0 if the query has no gold in the pool)"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = []
    for i, q in enumerate(qids):
        g = [cidx[c] for c in gold[q] if c in cidx and c != q]
        out.append(int((S[i] > S[i, g].max()).sum()) + 1 if g else 0)
    return out


# ---------------------------------------------------------------------------------------------
# BM25 (classical lexical ranking; fitted on the evaluated pool only - no pretrained model involved)
# ---------------------------------------------------------------------------------------------
def bm25_scores(q_texts, c_texts, ngram=(1, 1), k1=1.2, b=0.75):
    """Okapi BM25 with Lucene-style idf and sub-linear query term frequency.  -> [n_queries, n_candidates] float32"""
    import scipy.sparse as sps
    from sklearn.feature_extraction.text import CountVectorizer
    vec = CountVectorizer(lowercase=True, ngram_range=tuple(ngram), token_pattern=r"(?u)\b\w\w+\b",
                          dtype=np.float32, max_features=2_000_000)
    try:
        X = vec.fit_transform(c_texts).tocsr()
    except ValueError:                                              # empty vocabulary
        return np.zeros((len(q_texts), len(c_texts)), dtype=np.float32)
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel()
    avgdl = max(float(dl.mean()), 1.0)
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = X.data
    X.data = (tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * dl[rows] / avgdl))).astype(np.float32)
    X = (X @ sps.diags(idf)).tocsr()
    Q = vec.transform(q_texts).tocsr()
    Q.data = (1.0 + np.log(Q.data)).astype(np.float32)
    return (Q @ X.T).toarray().astype(np.float32)


# ---------------------------------------------------------------------------------------------
# legal references: statute sections, constitutional articles, named Acts / Codes
# ---------------------------------------------------------------------------------------------
_NUMLIST = r"\d+[A-Za-z]?(?:\s*\(\w+\))*(?:\s*(?:,|and|&|or|to|/)\s*\d+[A-Za-z]?(?:\s*\(\w+\))*)*"
_SEC = re.compile(r"\b(?:sections?|secs?\.?|s\.)\s*(" + _NUMLIST + ")", re.I)
_ART = re.compile(r"\barticles?\s*(" + _NUMLIST + ")", re.I)
_ORD = re.compile(r"\border\s+([ivxlc]+|\d+)\s*,?\s*rules?\s*(\d+[A-Za-z]?)", re.I)
_ACT = re.compile(r"\b((?:[A-Z][A-Za-z\.&']*\s+){1,6}(?:Act|Code|Constitution))\b")
_ABBR = re.compile(r"\b(IPC|CrPC|Cr\.P\.C|CPC|C\.P\.C|NDPS|PMLA|UAPA|POCSO|TADA|MACT)\b")
_NUM = re.compile(r"\d+[A-Za-z]?")


def legal_refs(text):
    toks = []
    for m in _SEC.finditer(text):
        toks += ["sec" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ART.finditer(text):
        toks += ["art" + n.lower() for n in _NUM.findall(m.group(1))]
    for m in _ORD.finditer(text):
        toks.append(f"ord{m.group(1).lower()}r{m.group(2).lower()}")
    for m in _ACT.finditer(text):
        name = re.sub(r"^(?:the|this|that|of|in|under|by)\s+", "", m.group(1).strip(), flags=re.I)
        name = re.sub(r"\s+", "_", name.lower())
        if len(name) > 4:
            toks.append("act:" + name)
    for m in _ABBR.finditer(text):
        toks.append("act:" + m.group(1).lower().replace(".", ""))
    return " ".join(toks)


def refs_scores(q_texts, c_texts):
    """TF-IDF cosine over the extracted legal-reference tokens only. [n_queries, n_candidates] float32"""
    from sklearn.feature_extraction.text import TfidfVectorizer
    cr = [legal_refs(t) for t in c_texts]
    qr = [legal_refs(t) for t in q_texts]
    vec = TfidfVectorizer(sublinear_tf=True, token_pattern=r"\S+", lowercase=False, min_df=1, dtype=np.float32)
    try:
        X = vec.fit_transform(cr)
    except ValueError:
        return np.zeros((len(q_texts), len(c_texts)), dtype=np.float32)
    return (vec.transform(qr) @ X.T).toarray().astype(np.float32)


# ---------------------------------------------------------------------------------------------
# fusion of score components (each component z-scored per query), weights tuned on dev
# ---------------------------------------------------------------------------------------------
def fuse(comps, weights):
    out = None
    for n, w in weights.items():
        if w == 0 or n not in comps:
            continue
        out = w * comps[n] if out is None else out + w * comps[n]
    return out


DEFAULT_GRID = {"q2c": (0, 0.25, 0.5, 1, 2), "c2q": (0, 0.25, 0.5, 1, 2),
                "bm25": (0, 0.25, 0.5, 1, 2, 4, 8), "refs": (0, 0.1, 0.25, 0.5, 1)}


def tune_fusion(comps, primary, sp, k, grid=None, min_gain=0.002, passes=2):
    """
    greedy coordinate ascent on dev micro-F1@k.  `primary` keeps weight 1; every other component starts at 0 and is only
    switched on when it improves dev F1 by at least `min_gain` (guards against fitting noise on a small dev set).
    """
    grid = grid or DEFAULT_GRID
    weights = {n: 0.0 for n in comps}
    weights[primary] = 1.0
    best = base = f1_of(fuse(comps, weights), sp, k)
    for _ in range(passes):
        changed = False
        for n in comps:
            if n == primary:
                continue
            cur = weights[n]
            cand_best, cand_w = best, cur
            for w in grid.get(n, (0, 0.25, 0.5, 1, 2)):
                if w == cur:
                    continue
                trial = dict(weights)
                trial[n] = w
                f = f1_of(fuse(comps, trial), sp, k)
                if f > cand_best + (min_gain if cur == 0 else 1e-9):
                    cand_best, cand_w = f, w
            if cand_w != cur:
                weights[n], best, changed = cand_w, cand_best, True
        if not changed:
            break
    return weights, best, base


def prf_component(S, Ec, qids, cids, m):
    """
    pseudo-relevance feedback in embedding space: centroid of the m best candidates of each query (query itself excluded),
    scored against every candidate.  Relevant cases of one query tend to be mutually similar (they are co-cited).
    """
    Sm = mask_self(S, qids, cids)
    m = min(m, Sm.shape[1] - 1)
    top = np.argpartition(-Sm, m - 1, axis=1)[:, :m]
    cent = l2n(Ec[top].mean(1))
    return zrows(cent @ Ec.T)


def tune_prf(S, Ec, sp, k, ms=(3, 5, 8), gammas=(0.25, 0.5, 1.0), min_gain=0.002):
    base = f1_of(S, sp, k)
    best = (base, None, 0.0)
    for m in ms:
        P = prf_component(S, Ec, sp.qids, sp.cids, m)
        for g in gammas:
            f = f1_of(S + g * P, sp, k)
            if f > best[0]:
                best = (f, m, g)
    if best[1] is None or best[0] < base + min_gain:
        return None, 0.0, base, base
    return best[1], best[2], best[0], base


def mine_hard_negs(S, qids, cids, gold, pool_n, skip=0):
    """per query: ranks [skip, skip+pool_n) of the candidates that are neither the query itself nor gold"""
    S = mask_self(S, qids, cids)
    cidx = {c: i for i, c in enumerate(cids)}
    out = {}
    for i, q in enumerate(qids):
        for g in gold[q]:
            j = cidx.get(g)
            if j is not None:
                S[i, j] = -np.inf
        n = min(skip + pool_n, S.shape[1] - 1)
        idx = np.argpartition(-S[i], n - 1)[:n]
        idx = idx[np.argsort(-S[i, idx])][skip:]
        out[q] = [cids[j] for j in idx if np.isfinite(S[i, j])]
    return out


def per_query_table(S, sp, preds, k):
    ranks = first_hit_rank(S, sp.qids, sp.cids, sp.gold)
    rows = []
    for i, q in enumerate(sp.qids):
        g = sp.gold[q]
        hits = len(set(preds[q][:k]) & g)
        rows.append({"qid": q, "n_gold": len(g), "hits": hits,
                     "precision": hits / k, "recall": hits / len(g) if g else 0.0,
                     "first_hit_rank": ranks[i], "q_tokens": len(sp.qdocs[i].ids),
                     "q_chunks": len(sp.qdocs[i].spans)})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- self-tests
def _selftest_retrieval():
    preds = {"a": ["1", "2", "3", "4", "5", "6", "7"], "b": ["1", "2", "3", "4", "5", "6", "7"]}
    gold = {"a": {"1", "2", "9"}, "b": {"7"}}
    m = micro_prf(preds, gold, ["a", "b"], 7)
    assert (m["tp"], m["n_pred"], m["n_gold"]) == (3, 14, 4)
    assert abs(m["F1"] - 2 * (3 / 14) * (3 / 4) / (3 / 14 + 3 / 4)) < 1e-12
    S = np.array([[0.9, 0.1, 0.5, 0.8], [0.2, 0.3, 0.1, 0.95]], dtype=np.float32)
    top = rank_topk(S, ["x", "y"], ["x", "b", "c", "d"], 2)          # query "x" is in the pool at column 0
    assert top["x"] == ["d", "c"] and top["y"] == ["d", "b"], top
    # hard negatives: self and gold removed, optional skip window
    cids = ["x", "g1", "n1", "n2", "n3", "n4"]
    S = np.array([[0.99, 0.95, 0.9, 0.5, 0.4, 0.3]], dtype=np.float32)
    assert mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2)["x"] == ["n1", "n2"]
    assert mine_hard_negs(S, ["x"], cids, {"x": {"g1"}}, 2, skip=1)["x"] == ["n2", "n3"]
    # BM25 prefers the document that shares rare terms; legal reference extraction
    docs = ["tenant eviction under rent control act section 13", "murder trial under section 302 ipc",
            "property partition suit among brothers", "bail application under section 439 crpc"]
    sc = bm25_scores(["eviction of tenant rent control"], docs)[0]
    assert int(np.argmax(sc)) == 0 and sc.shape == (4,)
    r = legal_refs("Under Section 302 and 34 of the Indian Penal Code read with Article 21 of the Constitution and IPC")
    assert "sec302" in r and "sec34" in r and "art21" in r and "act:ipc" in r, r
    rs = refs_scores(["section 302 IPC murder"], docs)[0]
    assert int(np.argmax(rs)) == 1, rs
    # fusion: a weak primary + a good lexical component -> the tuner switches the lexical one on
    rng = np.random.default_rng(0)
    nq, nc = 20, 120
    cid = [f"c{i}" for i in range(nc)]
    qid = [f"q{i}" for i in range(nq)]
    gold = {q: set(rng.choice(cid, 5, replace=False)) for q in qid}
    good = rng.normal(size=(nq, nc)).astype(np.float32)
    weak = rng.normal(size=(nq, nc)).astype(np.float32)
    for i, q in enumerate(qid):
        for g in gold[q]:
            good[i, int(g[1:])] += 3.0
            weak[i, int(g[1:])] += 0.3
    sp_ = types.SimpleNamespace(qids=qid, cids=cid, gold=gold)
    comps = {"pooled": zrows(weak), "bm25": zrows(good)}
    w, best, base = tune_fusion(comps, "pooled", sp_, 7)
    assert w["bm25"] > 0 and best > base + 0.1, (w, best, base)
    # PRF returns a valid z-scored matrix of the right shape
    E = l2n(rng.normal(size=(nc, 16)).astype(np.float32))
    assert prf_component(good, E, qid, cid, 3).shape == (nq, nc)
    return True


_selftest_retrieval()

### 5b. Extra post-processing signals
Pool-structure diagnostics, cross-query competition (Sinkhorn / mutual nearest neighbour), temporal prior learned on train, BM25 on shortened queries, the tuned-recipe machinery and a cross-validated estimate. Numpy / scipy / scikit-learn only; self-test at import.

In [ ]:
# =============================================================================
# CELL 5b - extra post-processing signals (numpy / scipy / scikit-learn only; no pretrained model, no labels of dev/test)
#
#   1. pool-structure diagnostics        how many queries is each pool case gold for?  (printed, so you can see the structure)
#   2. competition between queries       a pool case is usually cited by few queries, so it should not be every query's top hit:
#                                        Sinkhorn / dual-softmax hub penalty, or "mutual nearest neighbour" filter.  TRANSDUCTIVE:
#                                        it looks at all queries of the split together (no labels).  Switch off: USE_COMPETE=False
#   3. temporal prior                    a prior case cannot be decided after the query.  Years found in the texts + the gap
#                                        distribution of gold pairs LEARNED ON TRAIN -> log-likelihood-ratio matrix
#   4. BM25 on shortened queries         long queries saturate BM25: keep each query's `top` highest tf*idf terms
#   5. tune_recipe / apply_recipe        dev-tuned fusion -> PRF -> competition -> refinement, applied identically to test
#   6. cv_estimate                       5-fold cross-validated estimate of the whole tuning procedure on dev (honest number)
#   8. citation metadata                 the report header "CITATION: 1964 AIR 1545 ... CITATOR INFO : R 1970 SC 765 (9) ..." lists LATER cases that
#                                        cite this case.  If a query cites a candidate, the query's own citation usually sits in the candidate's
#                                        CITATOR list (back-link) and both share later citing cases (co-citation).  Read from the texts only.
#   7. refinement (new)                  competition <-> embedding feedback, repeated: the competition makes each query's top set
#                                        cleaner, the centroid of that cleaner set is a better query for the next round, and the next
#                                        competition settles the conflicts that centroid creates.  Transductive like the competition.
# Every extra only stays on if it gains >= MIN_GAIN dev F1@K.
# =============================================================================
from scipy.special import logsumexp as _lse


# ---------------------------------------------------------------------------------------------- 1. diagnostics
def gold_structure(sp):
    cnt = Counter(c for q in sp.qids for c in sp.gold_in_pool[q])
    sizes = [len(sp.gold[q]) for q in sp.qids]
    n_c = len(sp.cids)
    return {"split": getattr(sp, "name", "?"), "queries": len(sp.qids), "pool": n_c,
            "total_gold": int(sum(sizes)), "gold_in_pool": int(sum(len(sp.gold_in_pool[q]) for q in sp.qids)),
            "pool_gold_for_0_queries": int(n_c - len(cnt)),
            "pool_gold_for_1_query": int(sum(1 for v in cnt.values() if v == 1)),
            "pool_gold_for_2plus": int(sum(1 for v in cnt.values() if v >= 2)),
            "max_queries_per_case": int(max(cnt.values())) if cnt else 0}


# ---------------------------------------------------------------------------------------------- 2. competition
def _z_finite(S):
    S = S.astype(np.float32, copy=True)
    fin = np.isfinite(S)
    cnt = np.maximum(fin.sum(1, keepdims=True), 1)
    mu = np.where(fin, S, 0.0).sum(1, keepdims=True) / cnt
    var = (np.where(fin, S - mu, 0.0) ** 2).sum(1, keepdims=True) / cnt
    out = (S - mu) / np.sqrt(var + 1e-6)
    out[~fin] = -np.inf
    return out.astype(np.float32)


def compete(S, qids, cids, mode="sinkhorn", tau=1.0, lam=1.0, iters=1, m=1, pen=1.0):
    """
    re-score S [n_queries, n_candidates] using the other queries of the same split (no labels).
      sinkhorn : alternate  row log-softmax  and  column penalty  lam * tau * logsumexp_q(S/tau)   (lam=1 = dual softmax;
                 lam<0 would boost popular cases instead of penalising them)
      mnn      : subtract `pen` where the query is not among the `m` best-matching queries of that candidate
    The query's own case stays at -inf.
    """
    X = _z_finite(mask_self(S, qids, cids))
    if mode == "sinkhorn":
        for _ in range(int(iters)):
            X = X - tau * _lse(X / tau, axis=1, keepdims=True)
            cl = tau * _lse(X / tau, axis=0, keepdims=True)
            X = X - lam * np.where(np.isfinite(cl), cl, 0.0)
        return X.astype(np.float32)
    if mode == "mnn":
        order = np.argsort(-np.where(np.isfinite(X), X, -1e9), axis=0, kind="stable")      # queries by score, per candidate
        rank = np.empty_like(order)
        np.put_along_axis(rank, order, np.broadcast_to(np.arange(X.shape[0])[:, None], order.shape), axis=0)
        return (X - pen * (rank >= m)).astype(np.float32)
    raise ValueError(mode)


COMPETE_GRID = ([{"mode": "sinkhorn", "tau": t, "lam": l, "iters": i}
                 for t in (0.2, 0.3, 0.5, 1.0) for l in (-0.5, 0.5, 1.0) for i in (1, 2, 3, 5)]
                + [{"mode": "mnn", "m": m, "pen": p} for m in (1, 2, 3) for p in (0.5, 1.0, 2.0, 4.0)])


def _rows_counts(S_rows, qids_rows, cids, gold, k):
    m = micro_prf(rank_topk(S_rows, qids_rows, cids, k), gold, qids_rows, k)
    return m["tp"], m["n_pred"], m["n_gold"]


def _f1_from_counts(tp, n_pred, n_gold):
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    return 2 * P * R / (P + R) if P + R > 0 else 0.0


def _best_compete(S, qids, cids, gold, rows, k, min_gain, grid=None):
    """tune the competition step on the query rows `rows` (indices); the transform itself always sees all queries"""
    rows = np.asarray(rows)
    qrows = [qids[i] for i in rows]
    base = _f1_from_counts(*_rows_counts(S[rows], qrows, cids, gold, k))
    best, best_f = None, base
    for p in (grid or COMPETE_GRID):
        X = compete(S, qids, cids, **p)
        f = _f1_from_counts(*_rows_counts(X[rows], qrows, cids, gold, k))
        if f > best_f:
            best, best_f = p, f
    if best is None or best_f < base + min_gain:
        return None, base, base
    return best, best_f, base


def tune_compete(S, sp, k, min_gain=0.01):
    # stricter than the fusion guard: ~36 settings are tried, and the best of them gains a little on pure noise
    return _best_compete(S, sp.qids, sp.cids, sp.gold, np.arange(len(sp.qids)), k, min_gain)


# ---------------------------------------------------------------------------------------------- 2b. refinement
REFINE_MS = (4, 6, 8, 12)
REFINE_GAMMAS = (0.5, 1.0, 2.0)
REFINE_ROUNDS = 4


def refine_scores(S, Ce, qids, cids, comp, m, gamma, rounds, trace=False):
    """competition -> [ centroid of each query's m best cases (embedding space) -> add gamma * its similarity -> competition ] x rounds.
    The competition parameters `comp` are the ones tuned for the plain competition.  trace=True returns the list of the score
    matrices after round 1, 2, ... (so that every number of rounds can be evaluated from one pass).  No labels are used."""
    X = compete(S, qids, cids, **comp)
    outs = []
    for _ in range(int(rounds)):
        Y = _z_finite(X) + gamma * prf_component(X, Ce, qids, cids, int(m))
        X = compete(Y, qids, cids, **comp)
        outs.append(X)
    return outs if trace else X


def tune_refine(S, Ce, sp, k, comp, min_gain=0.01, rows=None, ms=None, gammas=None, rounds=None):
    """S: scores BEFORE the competition (fusion + PRF).  Returns (refine dict | None, F1 with refine, F1 of the plain competition)"""
    ms, gammas, rounds = ms or REFINE_MS, gammas or REFINE_GAMMAS, rounds or REFINE_ROUNDS
    rows = np.arange(len(sp.qids)) if rows is None else np.asarray(rows)
    qrows = [sp.qids[i] for i in rows]
    base = _f1_from_counts(*_rows_counts(compete(S, sp.qids, sp.cids, **comp)[rows], qrows, sp.cids, sp.gold, k))
    best, best_f = None, base
    for m in ms:
        for g in gammas:
            for r, X in enumerate(refine_scores(S, Ce, sp.qids, sp.cids, comp, m, g, rounds, trace=True), 1):
                f = _f1_from_counts(*_rows_counts(X[rows], qrows, sp.cids, sp.gold, k))
                if f > best_f:
                    best, best_f = {"m": int(m), "gamma": float(g), "rounds": int(r)}, f
    if best is None or best_f < base + min_gain:
        return None, base, base
    return best, best_f, base


# ---------------------------------------------------------------------------------------------- 8. citation metadata
_YR = r"(1[89]\d\d|20[0-2]\d)"
_OWN_RX = [("a", re.compile(r"(?<!\d)" + _YR + r"\s+AIR\s+(?:SC\s+)?(\d{1,5})(?!\d)")),
           ("a", re.compile(r"\bAIR\s+" + _YR + r"\s+(?:SC\s+)?(\d{1,5})(?!\d)")),
           ("r", re.compile(r"(?<!\d)" + _YR + r"\s+SCR\s*(?:\(\s*\d+\s*\)\s*)?(?:Supp\.?\s*)?(\d{1,5})(?!\d)", re.I)),
           ("c", re.compile(r"(?<!\d)\(?" + _YR + r"\)?\s*(?:\d+\s+)?SCC\s+(\d{1,5})(?!\d)"))]
_FAMILY = {"SC": "a", "AIR": "a", "SCR": "r", "SCC": "c"}
_CITATOR_RX = re.compile(r"(?<![A-Za-z])[A-Z]{1,2}\s+" + _YR + r"\s+(SC|SCR|SCC|AIR)\s*(?:\(\s*\d+\s*\)\s*)?(\d{1,5})(?!\d)")
_CITATION_LABEL = re.compile(r"CITATION\s*:")
_CITATOR_LABEL = re.compile(r"CITATOR\s+INFO\s*:")
_CITATOR_END = re.compile(r"\b(?:ACT|HEADNOTE|JUDGMENT|APPEAL|PETITION)\s*:")


def parse_header(text, head_chars=8000):
    """-> (own_keys, citator_keys) of one case text.  A key is '<family><year>:<page>' (a = AIR / SC, r = SCR, c = SCC), so the
    query's 'CITATION: 1964 AIR 1545' and a candidate's 'CITATOR INFO : R 1964 SC 1545 (9)' give the same key 'a1964:1545'."""
    h = text[:head_chars]
    own, cit = set(), set()
    m, m2 = _CITATION_LABEL.search(h), _CITATOR_LABEL.search(h)
    if m:
        end = m2.start() if (m2 is not None and m2.start() > m.end()) else m.end() + 300
        seg = h[m.end():end]
        for fam, rx in _OWN_RX:
            for g in rx.finditer(seg):
                own.add(f"{fam}{g.group(1)}:{int(g.group(2))}")
    if m2:
        seg = h[m2.end():]
        e = _CITATOR_END.search(seg)
        if e:
            seg = seg[:e.start()]
        for g in _CITATOR_RX.finditer(seg):
            cit.add(f"{_FAMILY[g.group(2)]}{g.group(1)}:{int(g.group(3))}")
    return own, cit


def cite_matrices(q_texts, c_texts):
    """-> dict(cite, cocite, cpop): float32 [n_queries, n_candidates] and a stats dict.
      cite   : number of the query's own citation keys that appear in the candidate's CITATOR list      (back-link)
      cocite : cosine of the idf-weighted CITATOR sets of query and candidate                           (co-citation by later cases)
      cpop   : log(1 + length of the candidate's CITATOR list)                                          (how often it was cited later)
    a query's own copy in the pool is zeroed in `cocite` (identical text)"""
    import scipy.sparse as sps
    pq, pc = [parse_header(t) for t in q_texts], [parse_header(t) for t in c_texts]
    nq, nc = len(pq), len(pc)
    inv = defaultdict(list)
    for j, (_, cit) in enumerate(pc):
        for k in cit:
            inv[k].append(j)
    cite = np.zeros((nq, nc), np.float32)
    for i, (own, _) in enumerate(pq):
        for k in own:
            for j in inv.get(k, ()):
                cite[i, j] += 1.0
    vocab, df = {}, Counter()
    for _, cit in pq + pc:
        for k in cit:
            vocab.setdefault(k, len(vocab))
            df[k] += 1
    cocite = np.zeros((nq, nc), np.float32)
    if vocab:
        N = nq + nc
        def mat(parsed):
            r, c, v = [], [], []
            for i, (_, cit) in enumerate(parsed):
                for k in cit:
                    r.append(i), c.append(vocab[k]), v.append(math.log((N + 1.0) / (df[k] + 1.0)) + 1.0)
            M = sps.csr_matrix((v, (r, c)), shape=(len(parsed), len(vocab)), dtype=np.float32)
            nrm = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()) + 1e-9
            return sps.diags(1.0 / nrm) @ M
        cocite = (mat(pq) @ mat(pc).T).toarray().astype(np.float32)
        where = {}
        for j, t in enumerate(c_texts):
            where.setdefault(hash(t), j)
        for i, t in enumerate(q_texts):
            j = where.get(hash(t))
            if j is not None and c_texts[j] == t:
                cocite[i, j] = 0.0
    pop = np.log1p(np.array([len(cit) for _, cit in pc], np.float32))
    stats = {"queries_with_own_key": float(np.mean([bool(o) for o, _ in pq])) if nq else 0.0,
             "queries_with_citator": float(np.mean([bool(c) for _, c in pq])) if nq else 0.0,
             "cands_with_own_key": float(np.mean([bool(o) for o, _ in pc])) if nc else 0.0,
             "cands_with_citator": float(np.mean([bool(c) for _, c in pc])) if nc else 0.0,
             "median_citator_len": float(np.median([len(c) for _, c in pc if c])) if any(c for _, c in pc) else 0.0}
    return {"cite": cite, "cocite": cocite, "cpop": np.broadcast_to(pop[None, :], (nq, nc)).copy()}, stats


def cite_diagnostics(M, stats, sp):
    """how good is each citation component on the labelled split `sp`?  precision / recall of its non-zero pairs against the gold"""
    cidx = {c: j for j, c in enumerate(sp.cids)}
    gold_cells = np.zeros(M["cite"].shape, bool)
    for i, q in enumerate(sp.qids):
        for c in sp.gold_in_pool[q]:
            if c in cidx:
                gold_cells[i, cidx[c]] = True
    row = {"split": getattr(sp, "name", "?"), **{k: round(v, 3) for k, v in stats.items()}}
    for k in ("cite", "cocite"):
        nz = M[k] > 0
        n_nz, tp = int(nz.sum()), int((nz & gold_cells).sum())
        row[f"{k}_pairs"] = n_nz
        row[f"{k}_precision"] = round(tp / n_nz, 3) if n_nz else 0.0
        row[f"{k}_recall"] = round(tp / max(int(gold_cells.sum()), 1), 3)
    row["base_rate"] = round(float(gold_cells.mean()), 5)
    return row



# ---------------------------------------------------------------------------------------------- 3. temporal prior
_YEAR = re.compile(r"(?<!\d)((?:185|186|187|188|189|190|191|192|193|194|195|196|197|198|199|200|201|202)\d)(?!\d)")
GAP_LO, GAP_HI = -60, 60
YEAR_MODES = ("max", "head", "p90")


def years_of(texts, mode="max", head_chars=3000):
    """estimated decision year of every text (NaN = no year found).  max: latest year mentioned; head: latest year in the
    first `head_chars` characters; p90: 90th percentile of all years mentioned"""
    out = np.full(len(texts), np.nan, dtype=np.float32)
    for i, t in enumerate(texts):
        ys = [int(y) for y in _YEAR.findall(t[:head_chars] if mode == "head" else t)]
        if ys:
            out[i] = max(ys) if mode in ("max", "head") else float(np.rint(np.percentile(ys, 90)))
    return out


def fit_gap_prior(yq, yc, gold_pairs, alpha=1.0):
    """log-likelihood ratio  log p(gap | gold pair) / p(gap | any pair)  for gap = year(query) - year(candidate)
    (positive gap = the candidate is older).  Learned on TRAIN.  Returns (lr over gap bins, KL information in nats)."""
    B = GAP_HI - GAP_LO + 1
    g_hist, a_hist = np.zeros(B), np.zeros(B)
    for qi, ci in gold_pairs:
        if np.isfinite(yq[qi]) and np.isfinite(yc[ci]):
            g_hist[int(np.clip(yq[qi] - yc[ci], GAP_LO, GAP_HI)) - GAP_LO] += 1
    ycv = yc[np.isfinite(yc)]
    for qi in range(len(yq)):
        if np.isfinite(yq[qi]) and len(ycv):
            a_hist += np.bincount(np.clip(yq[qi] - ycv, GAP_LO, GAP_HI).astype(int) - GAP_LO, minlength=B)
    ker = np.array([0.25, 0.5, 0.25])
    for _ in range(2):
        g_hist, a_hist = np.convolve(g_hist, ker, "same"), np.convolve(a_hist, ker, "same")
    pg = (g_hist + alpha) / (g_hist.sum() + alpha * B)
    pa = (a_hist + alpha) / (a_hist.sum() + alpha * B)
    lr = np.log(pg / pa)
    return np.clip(lr, -5, 5).astype(np.float32), float((pg * lr).sum())


def gap_matrix(lr, yq, yc):
    gap = np.clip(yq[:, None] - yc[None, :], GAP_LO, GAP_HI)
    ok = np.isfinite(gap)
    M = np.zeros(gap.shape, dtype=np.float32)
    M[ok] = lr[gap[ok].astype(int) - GAP_LO]
    return M


def fit_time_prior(sp, min_kl=0.03):
    """learn the year-gap prior on the train split, pick the year definition with the most information.
    -> dict(mode, lr, kl, coverage) or None when the texts carry no usable temporal signal"""
    qt = [sp.qtext[q] for q in sp.qids]
    ct = [sp.ctext[c] for c in sp.cids]
    cidx = {c: i for i, c in enumerate(sp.cids)}
    pairs = [(qi, cidx[c]) for qi, q in enumerate(sp.qids) for c in sp.gold_in_pool[q] if c in cidx]
    best, rows = None, []
    for mode in YEAR_MODES:
        yq, yc = years_of(qt, mode), years_of(ct, mode)
        lr, kl = fit_gap_prior(yq, yc, pairs)
        cov = float(np.isfinite(yc).mean())
        rows.append({"year_definition": mode, "coverage": round(cov, 3), "kl_nats": round(kl, 4)})
        if best is None or kl > best["kl"]:
            best = {"mode": mode, "lr": lr, "kl": kl, "coverage": cov}
    log("[time prior] information of the year gap about 'is this a gold pair' (train), per year definition:")
    log(pd.DataFrame(rows).to_string(index=False))
    return best if best["kl"] >= min_kl and best["coverage"] >= 0.3 else None


def time_component(prior, q_texts, c_texts):
    yq, yc = years_of(q_texts, prior["mode"]), years_of(c_texts, prior["mode"])
    return zrows(gap_matrix(prior["lr"], yq, yc))


# ---------------------------------------------------------------------------------------------- 4. BM25, shortened queries
def _bm25_parts(q_texts, c_texts, ngram, k1, b):
    """doc-term BM25 weights (idf included) and the sub-linear query term frequencies"""
    import scipy.sparse as sps
    from sklearn.feature_extraction.text import CountVectorizer
    vec = CountVectorizer(lowercase=True, ngram_range=tuple(ngram), token_pattern=r"(?u)\b\w\w+\b",
                          dtype=np.float32, max_features=2_000_000)
    try:
        X = vec.fit_transform(c_texts).tocsr()
    except ValueError:
        return None
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel()
    avgdl = max(float(dl.mean()), 1.0)
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = X.data
    X.data = (tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * dl[rows] / avgdl))).astype(np.float32)
    X = (X @ sps.diags(idf)).tocsr()
    Q = vec.transform(q_texts).tocsr()
    Q.data = (1.0 + np.log(Q.data)).astype(np.float32)
    return X, Q, idf


def _keep_top_terms(Q, idf, top):
    Q = Q.copy()
    if top:
        w = Q.data * idf[Q.indices]
        for i in range(Q.shape[0]):
            s, e = Q.indptr[i], Q.indptr[i + 1]
            if e - s > top:
                thr = np.partition(w[s:e], -top)[-top]
                Q.data[s:e][w[s:e] < thr] = 0.0
        Q.eliminate_zeros()
    return Q


def bm25_topterms_multi(q_texts, c_texts, ngram=(1, 1), k1=1.2, b=0.75, tops=(100,)):
    """BM25 where each query keeps only its `top` terms with the largest (1+log tf) * idf.  One index build for all `tops`.
    top=None keeps every term (= bm25_scores).  -> {top: [n_queries, n_candidates] float32}"""
    parts = _bm25_parts(q_texts, c_texts, ngram, k1, b)
    if parts is None:
        return {t: np.zeros((len(q_texts), len(c_texts)), dtype=np.float32) for t in tops}
    X, Q, idf = parts
    return {t: (_keep_top_terms(Q, idf, t) @ X.T).toarray().astype(np.float32) for t in tops}


def bm25_topterms(q_texts, c_texts, ngram=(1, 1), k1=1.2, b=0.75, top=100):
    return bm25_topterms_multi(q_texts, c_texts, ngram, k1, b, (top,))[top]


# ---------------------------------------------------------------------------------------------- 5. recipe: tune on dev, apply to test
GRID_V3 = {**DEFAULT_GRID, "bm25_top": (0, 0.25, 0.5, 1, 2, 4, 8), "time": (0, 0.1, 0.25, 0.5, 1, 2),
           "cite": (0, 0.05, 0.1, 0.25, 0.5, 1, 2), "cocite": (0, 0.1, 0.25, 0.5, 1, 2, 4),
           "cpop": (0, 0.1, 0.25, 0.5, 1), "tprior": (0, 0.1, 0.25, 0.5, 1)}


def tune_recipe(Z, primary, Ce, sp, k, use_prf=True, use_compete=True, min_gain=0.002, grid=None, compete_min_gain=0.01,
                use_refine=True):
    """Z: dict name -> z-scored [n_q, n_c] components.  Tuned on dev: fusion weights -> PRF -> competition -> refinement.
    Returns (recipe, final dev score matrix)"""
    W, f_fused, f_primary = tune_fusion(Z, primary, sp, k, grid=grid or GRID_V3, min_gain=min_gain)
    S = fuse(Z, W)
    rec = {"weights": W, "prf_m": None, "prf_gamma": 0.0, "compete": None, "refine": None,
           "dev_f1": {"primary_alone": f_primary, "fused": f_fused, "fused_prf": f_fused, "competition": f_fused, "final": f_fused}}
    if use_prf and Ce is not None:
        m, g, f_prf, _ = tune_prf(S, Ce, sp, k, min_gain=min_gain)
        if m is not None:
            S = S + g * prf_component(S, Ce, sp.qids, sp.cids, m)
            rec.update(prf_m=m, prf_gamma=g)
        rec["dev_f1"]["fused_prf"] = rec["dev_f1"]["final"] = f_prf
    if use_compete:
        p, f_c, f_before = tune_compete(S, sp, k, min_gain=compete_min_gain)
        rec["dev_f1"]["competition"] = rec["dev_f1"]["final"] = f_c
        if p is not None:
            S_pre = S
            S = compete(S_pre, sp.qids, sp.cids, **p)
            rec["compete"] = p
            if use_refine and Ce is not None:
                rf, f_r, _ = tune_refine(S_pre, Ce, sp, k, p, min_gain=2 * compete_min_gain)
                if rf is not None:
                    S = refine_scores(S_pre, Ce, sp.qids, sp.cids, p, rf["m"], rf["gamma"], rf["rounds"])
                    rec["refine"] = rf
                    rec["dev_f1"]["final"] = f_r
    return rec, S


def apply_recipe(Z, Ce, sp, rec):
    S = fuse(Z, rec["weights"])
    if rec["prf_m"] is not None:
        S = S + rec["prf_gamma"] * prf_component(S, Ce, sp.qids, sp.cids, rec["prf_m"])
    if rec["compete"] is not None:
        rf = rec.get("refine")
        if rf and Ce is not None:
            S = refine_scores(S, Ce, sp.qids, sp.cids, rec["compete"], rf["m"], rf["gamma"], rf["rounds"])
        else:
            S = compete(S, sp.qids, sp.cids, **rec["compete"])
    return S


def describe_recipe(rec):
    on = {n: w for n, w in rec["weights"].items() if w}
    rf = rec.get("refine")
    return f"weights {on} | PRF {'off' if rec['prf_m'] is None else 'top-%d x %.2f' % (rec['prf_m'], rec['prf_gamma'])} | " \
           f"competition {'off' if rec['compete'] is None else rec['compete']} | " \
           f"refinement {'off' if not rf else '%d rounds, top-%d x %.2f' % (rf['rounds'], rf['m'], rf['gamma'])}"


# ---------------------------------------------------------------------------------------------- 6. honest estimate
def cv_estimate(Z, primary, sp, k, folds=5, seed=0, min_gain=0.002, use_compete=True, grid=None, compete_min_gain=0.01,
                Ce=None, use_refine=False):
    """nested cross-validation over the dev QUERIES: fusion weights and competition parameters are tuned on the other folds
    and evaluated on the held-out fold (the competition transform itself never sees labels).  PRF is not included.
    -> micro-F1@k of the held-out predictions for: primary alone / tuned fusion / tuned fusion + competition"""
    n = len(sp.qids)
    perm = np.random.RandomState(seed).permutation(n)
    parts = np.array_split(perm, folds)
    tot = {"primary": [0, 0, 0], "fusion": [0, 0, 0], "fusion+competition": [0, 0, 0]}
    with_refine = bool(use_compete and use_refine and Ce is not None)
    if with_refine:
        tot["fusion+competition+refinement"] = [0, 0, 0]
    for f in range(folds):
        te = np.sort(parts[f])
        tr = np.sort(np.concatenate([parts[j] for j in range(folds) if j != f]))
        sub = types.SimpleNamespace(qids=[sp.qids[i] for i in tr], cids=sp.cids, gold=sp.gold)
        W, _, _ = tune_fusion({nme: v[tr] for nme, v in Z.items()}, primary, sub, k, grid=grid or GRID_V3, min_gain=min_gain)
        S = fuse(Z, W)
        qte = [sp.qids[i] for i in te]
        for name, M in (("primary", Z[primary]), ("fusion", S)):
            for j, c in enumerate(_rows_counts(M[te], qte, sp.cids, sp.gold, k)):
                tot[name][j] += c
        X = Xr = S
        if use_compete:
            p, _, _ = _best_compete(S, sp.qids, sp.cids, sp.gold, tr, k, compete_min_gain)
            if p is not None:
                X = Xr = compete(S, sp.qids, sp.cids, **p)
                if with_refine:
                    rf, _, _ = tune_refine(S, Ce, sp, k, p, min_gain=2 * compete_min_gain, rows=tr)
                    if rf is not None:
                        Xr = refine_scores(S, Ce, sp.qids, sp.cids, p, rf["m"], rf["gamma"], rf["rounds"])
        for j, c in enumerate(_rows_counts(X[te], qte, sp.cids, sp.gold, k)):
            tot["fusion+competition"][j] += c
        if with_refine:
            for j, c in enumerate(_rows_counts(Xr[te], qte, sp.cids, sp.gold, k)):
                tot["fusion+competition+refinement"][j] += c
    return {name: _f1_from_counts(*v) for name, v in tot.items()}


# ---------------------------------------------------------------------------------------------- self-test
def _selftest_extra():
    rng = np.random.default_rng(0)
    # --- BM25 with all terms == bm25_scores ; fewer terms differ
    docs = ["tenant eviction under rent control act section 13 landlord premises", "murder trial under section 302 ipc",
            "property partition suit among brothers", "bail application under section 439 crpc"]
    qs = ["eviction of tenant rent control landlord premises section 13"]
    assert np.allclose(bm25_topterms(qs, docs, top=None), bm25_scores(qs, docs), atol=1e-5)
    assert int(np.argmax(bm25_topterms(qs, docs, top=2)[0])) == 0
    # --- years
    assert years_of(["decided in 1999, cites 1975 and 2003 and Rs.2500"], "max")[0] == 2003
    assert years_of(["", "no digits"], "max").tolist()[0:1] != [0] and np.isnan(years_of([""], "max")[0])
    # --- competition on a one-to-one world: every pool case is gold for exactly one query
    nq, per = 60, 6
    nc = nq * per
    cids = [f"c{i}" for i in range(nc)]
    qids = [f"q{i}" for i in range(nq)]
    owner = np.repeat(np.arange(nq), per)
    gold = {q: set(cids[j] for j in np.where(owner == i)[0]) for i, q in enumerate(qids)}
    S = rng.normal(size=(nq, nc)).astype(np.float32)
    for i in range(nq):
        S[i, owner == i] += 1.6                                                   # moderate signal
    hubs = rng.choice(nc, 40, replace=False)
    S[:, hubs] += rng.normal(1.0, 0.3, size=(nq, 40)).astype(np.float32)           # a few hub cases similar to everyone
    sp_ = types.SimpleNamespace(qids=qids, cids=cids, gold=gold, gold_in_pool=gold, name="syn")
    base = f1_of(zrows(S), sp_, 6)
    p, f, b0 = tune_compete(zrows(S), sp_, 6)
    assert p is not None and f > base + 0.03 and abs(b0 - base) < 1e-9, (p, f, base)
    X = compete(zrows(S), qids, cids, **p)
    assert X.shape == S.shape and not np.isnan(X).any()
    # --- no structure (golds shared by all queries) -> the guard keeps competition off or harmless
    gold_shared = {q: set(cids[:6]) for q in qids}
    sp2 = types.SimpleNamespace(qids=qids, cids=cids, gold=gold_shared, gold_in_pool=gold_shared)
    S2 = rng.normal(size=(nq, nc)).astype(np.float32)
    S2[:, :6] += 2.0
    p2, f2, b2 = tune_compete(zrows(S2), sp2, 6)
    assert p2 is None or f2 >= b2, (p2, f2, b2)
    # --- temporal prior: gold cases are 1..15 years older than their query, other cases are anywhere
    yq = rng.integers(1995, 2022, size=nq).astype(np.float32)
    yc = rng.integers(1950, 2022, size=nc).astype(np.float32)
    for i in range(nq):
        yc[owner == i] = yq[i] - rng.integers(1, 16, size=per)
    lr, kl = fit_gap_prior(yq, yc, [(i, j) for i in range(nq) for j in np.where(owner == i)[0]])
    assert kl > 0.3 and lr[GAP_LO * -1 + 5] > 0 and lr[GAP_LO * -1 - 5] < 0, kl                # older = positive, newer = negative
    T = gap_matrix(lr, yq, yc)
    gold_cells = np.zeros((nq, nc), dtype=bool)
    for i in range(nq):
        gold_cells[i, owner == i] = True
    assert T.shape == (nq, nc) and T[gold_cells].mean() > T.mean() + 0.5, (T[gold_cells].mean(), T.mean())
    # --- recipe round trip + cross-validated estimate run and are sane
    Z = {"pooled": zrows(S), "time": zrows(T)}
    rec, Sd = tune_recipe(Z, "pooled", None, sp_, 6, use_prf=False)
    assert np.allclose(np.nan_to_num(apply_recipe(Z, None, sp_, rec), neginf=-1e9), np.nan_to_num(Sd, neginf=-1e9), atol=1e-4)
    assert rec["dev_f1"]["final"] >= rec["dev_f1"]["primary_alone"]
    cv = cv_estimate(Z, "pooled", sp_, 6, folds=4)
    assert set(cv) == {"primary", "fusion", "fusion+competition"} and all(0 <= v <= 1 for v in cv.values()), cv
    # --- refinement: cases cited by one query share a direction in embedding space -> refinement on; random embeddings -> off
    d = 24
    Ec = l2n((rng.standard_normal((nq, d))[owner] + 0.9 * rng.standard_normal((nc, d))).astype(np.float32))
    Zr = {"pooled": zrows(S)}
    rec_r, S_r = tune_recipe(Zr, "pooled", Ec, sp_, 6, use_prf=False)
    assert rec_r["compete"] is not None and rec_r["refine"] is not None, rec_r
    assert rec_r["dev_f1"]["final"] >= rec_r["dev_f1"]["competition"] + 0.02 and abs(f1_of(S_r, sp_, 6) - rec_r["dev_f1"]["final"]) < 1e-9
    assert np.allclose(np.nan_to_num(apply_recipe(Zr, Ec, sp_, rec_r), neginf=-1e9), np.nan_to_num(S_r, neginf=-1e9), atol=1e-4)
    assert "refinement" in describe_recipe(rec_r) and "rounds" in describe_recipe(rec_r)
    rec_x, _ = tune_recipe(Zr, "pooled", l2n(rng.standard_normal((nc, d)).astype(np.float32)), sp_, 6, use_prf=False)
    assert rec_x["refine"] is None, rec_x["refine"]                                       # no structure in the embeddings -> off
    # --- citation metadata: header parsing, back-link, co-citation
    hq = ("PETITIONER: Vs. RESPONDENT: X DATE OF JUDGMENT: 07/05/1963 CITATION: 1964 AIR 1545 1964 SCR (3) 573 "
          "CITATOR INFO : R 1970 SC 765 (9) R 1971 SC 342 (6) RF 1972 SCR (2) 40 (3) ACT: Constitution of India body text AIR 1950 SC 9")
    own, cit = parse_header(hq)
    assert own == {"a1964:1545", "r1964:573"}, own
    assert cit == {"a1970:765", "a1971:342", "r1972:40"}, cit
    assert parse_header("no header at all, AIR 1950 SC 9") == (set(), set())
    hc1 = "CITATION: 1950 AIR 99 CITATOR INFO : R 1964 SC 1545 (9) D 1970 SC 765 (2) ACT: x"           # cited by hq + shares 1970:765
    hc2 = "CITATION: 1951 AIR 7 CITATOR INFO : R 1980 SC 5 (1) ACT: x"                                # unrelated
    M, stt = cite_matrices([hq], [hc1, hc2, hq])
    assert M["cite"][0].tolist() == [1.0, 0.0, 0.0], M["cite"]
    assert M["cocite"][0, 0] > 0.2 and M["cocite"][0, 1] == 0.0 and M["cocite"][0, 2] == 0.0, M["cocite"]    # own copy zeroed
    assert M["cpop"].shape == (1, 3) and M["cpop"][0, 0] > M["cpop"][0, 1] > 0
    assert stt["cands_with_citator"] == 1.0 and stt["queries_with_own_key"] == 1.0
    sp_c = types.SimpleNamespace(name="syn", qids=["q"], cids=["a", "b", "q"], gold_in_pool={"q": ["a"]})
    dg = cite_diagnostics(M, stt, sp_c)
    assert dg["cite_precision"] == 1.0 and dg["cite_recall"] == 1.0, dg
    cv2 = cv_estimate(Zr, "pooled", sp_, 6, folds=4, Ce=Ec, use_refine=True)
    assert "fusion+competition+refinement" in cv2 and all(0 <= v <= 1 for v in cv2.values()), cv2
    return True


_selftest_extra()

### 5c. Lexical and temporal score components

In [ ]:
# =============================================================================
# CELL 5c - lexical / temporal components (shared by the dev evaluation, the test prediction and the rescoring notebook)
#   bm25      Okapi BM25 over the whole query
#   bm25_top  BM25 over each query's TOP-N terms (N chosen on dev among BM25_TOPS)
#   refs      overlap of statute sections / articles / Acts
#   time      temporal prior: log-likelihood ratio of the year gap, learned on TRAIN (switched off if the texts carry no years)
#   cite      back-link: the query's own report citation appears in the candidate's CITATOR INFO list (later cases citing it)
#   cocite    co-citation: query and candidate are cited by the same later cases (idf-weighted cosine of the CITATOR lists)
#   cpop      how long the candidate's CITATOR list is (how often it was cited later)
#   tprior    how often the candidate id is gold for a TRAIN query (only if dev/test pools share cases with train)
#   cite / cocite / cpop / tprior are read from the texts / train labels only; each is switched off when the data does not carry it
# =============================================================================
TIME_PRIOR = None
BM25_TOP_N = None
CITE_ON, TPRIOR_ON, TRAIN_POP = False, False, {}
LEX_NOTES = {}


def setup_lexical(dev):
    """fit the temporal prior on train and pick the BM25 top-N on dev.  Never raises: a failing extra is simply left out."""
    global TIME_PRIOR, BM25_TOP_N, CITE_ON, TPRIOR_ON, TRAIN_POP
    TIME_PRIOR, BM25_TOP_N = None, None
    CITE_ON, TPRIOR_ON, TRAIN_POP = False, False, {}
    if C.USE_TIME:
        try:
            TIME_PRIOR = fit_time_prior(SP["train"])
            log("[time prior] " + ("not used: the year gap carries too little information (or too few texts contain a year)"
                                   if TIME_PRIOR is None else
                                   f"year definition '{TIME_PRIOR['mode']}', information {TIME_PRIOR['kl']:.3f} nats, "
                                   f"{100 * TIME_PRIOR['coverage']:.0f}% of train candidates have a year"))
        except Exception as e:
            TIME_PRIOR = None
            log(f"!! temporal prior failed ({e!r}); continuing without it")
    if C.USE_BM25 and C.USE_BM25_TOP:
        try:
            dq, dc = [dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids]
            multi = bm25_topterms_multi(dq, dc, C.BM25_NGRAM, C.BM25_K1, C.BM25_B, tuple(C.BM25_TOPS) + (None,))
            rows = [{"query terms kept": ("all" if t is None else t), "F1@7": round(f1_of(S, dev, C.K), 4)} for t, S in multi.items()]
            log("[bm25 on shortened queries] dev F1@7 by number of query terms kept:")
            log(pd.DataFrame(rows).to_string(index=False))
            cand = [(f1_of(multi[t], dev, C.K), t) for t in C.BM25_TOPS]
            f_best, BM25_TOP_N = max(cand)
            if f_best <= f1_of(multi[None], dev, C.K):
                log("[bm25 on shortened queries] no better than the full query -> component not used")
                BM25_TOP_N = None
        except Exception as e:
            BM25_TOP_N = None
            log(f"!! BM25 on shortened queries failed ({e!r}); continuing without it")
    if getattr(C, "USE_CITE", False):
        try:
            rows = []
            for nm in ("train", "dev"):
                sp = SP[nm]
                M, st = cite_matrices([sp.qtext[q] for q in sp.qids], [sp.ctext[c] for c in sp.cids])
                rows.append(cite_diagnostics(M, st, sp))
                del M
            log("[citation metadata] report headers (CITATION / CITATOR INFO) read from the texts; precision / recall are those of the NON-ZERO pairs against the gold:")
            log(pd.DataFrame(rows).set_index("split").T.to_string())
            d = rows[-1]
            CITE_ON = d["cands_with_citator"] >= 0.05 and d["queries_with_own_key"] >= 0.05
            log("[citation metadata] " + ("components cite / cocite / cpop ON (the dev fusion decides their weights)" if CITE_ON else
                "not used: fewer than 5% of dev cases carry a parsable citation / citator list"))
        except Exception as e:
            CITE_ON = False
            log(f"!! citation metadata failed ({e!r}); continuing without it")
    if getattr(C, "USE_TPRIOR", False):
        try:
            TRAIN_POP = dict(Counter(c for q in SP["train"].qids for c in SP["train"].gold[q]))
            seen = np.array([c in TRAIN_POP for c in dev.cids])
            gold_pairs = [c for q in dev.qids for c in dev.gold_in_pool[q]]
            g_seen = float(np.mean([c in TRAIN_POP for c in gold_pairs])) if gold_pairs else 0.0
            log(f"[train popularity] {100 * seen.mean():.1f}% of the dev pool were gold for a train query; "
                f"{100 * g_seen:.1f}% of the dev gold pairs point at such a case")
            TPRIOR_ON = bool(seen.mean() >= 0.02)
            log("[train popularity] " + ("component tprior ON" if TPRIOR_ON else "not used: the dev pool shares (almost) no case with the train gold sets"))
        except Exception as e:
            TPRIOR_ON, TRAIN_POP = False, {}
            log(f"!! train popularity failed ({e!r}); continuing without it")
    LEX_NOTES.update(time_prior=None if TIME_PRIOR is None else TIME_PRIOR["mode"], bm25_top_n=BM25_TOP_N,
                     citation_metadata=bool(CITE_ON), train_popularity=bool(TPRIOR_ON))


def lexical_components(q_texts, c_texts, cids=None):
    out = {}
    if C.USE_BM25:
        out["bm25"] = bm25_scores(q_texts, c_texts, C.BM25_NGRAM, C.BM25_K1, C.BM25_B)
        if BM25_TOP_N:
            out["bm25_top"] = bm25_topterms(q_texts, c_texts, C.BM25_NGRAM, C.BM25_K1, C.BM25_B, BM25_TOP_N)
    if C.USE_REFS:
        out["refs"] = refs_scores(q_texts, c_texts)
    if TIME_PRIOR is not None:
        out["time"] = time_component(TIME_PRIOR, q_texts, c_texts)
    if CITE_ON:
        out.update(cite_matrices(q_texts, c_texts)[0])
    if TPRIOR_ON and cids is not None:
        pop = np.log1p(np.array([TRAIN_POP.get(c, 0) for c in cids], dtype=np.float32))
        out["tprior"] = np.broadcast_to(pop[None, :], (len(q_texts), len(c_texts))).copy()
    return out

### 6. Load the saved runs
Score files are aligned to this notebook's query / candidate order by id.

In [ ]:
# =============================================================================
# CELL 6 - rescoring: find the score files that earlier runs wrote and align them to the data of this notebook
#   each run folder holds   scores_dev.npz, scores_test.npz   (qids, cids, model, comp_*)
#                           dev_embeddings.npz, test_embeddings.npz   (optional; only needed for pseudo-relevance feedback)
#   Attach the output folders of your earlier runs (e.g. cs781_phase7_exp2_lormaplus_maxsim) as Kaggle inputs; they are found
#   automatically, or list them in CFG["RUN_DIRS"].  No GPU and no model are used in this notebook.
# =============================================================================
LEXICAL_NAMES = {"bm25", "bm25_top", "refs", "time"}      # saved by the earlier runs too, but recomputed here from the texts


def _ids(a):
    return [str(x) for x in np.asarray(a).tolist()]


def discover_runs():
    if C.RUN_DIRS:
        return [str(p) for p in C.RUN_DIRS]
    own, found = os.path.abspath(OUT_DIR), {}
    for root in (INPUT_ROOT, BASE_DIR):
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root, followlinks=True):
            if "scores_dev.npz" in files and "scores_test.npz" in files and os.path.abspath(d) != own:
                found.setdefault(os.path.realpath(d), d)
    return sorted(found.values())


def _align(sp, qids, cids, what):
    if set(qids) != set(map(str, sp.qids)) or set(cids) != set(map(str, sp.cids)):
        raise ValueError(f"{what}: its {len(qids)} queries / {len(cids)} candidates are not the ones loaded here "
                         f"({len(sp.qids)} / {len(sp.cids)}); was it a smoke run, or another split?")
    qi, ci = {q: i for i, q in enumerate(qids)}, {c: i for i, c in enumerate(cids)}
    return (np.array([qi[str(q)] for q in sp.qids]), np.array([ci[str(c)] for c in sp.cids]))


def load_run(d, sp, split):
    z = np.load(os.path.join(d, f"scores_{split}.npz"), allow_pickle=False)
    rq, rc = _align(sp, _ids(z["qids"]), _ids(z["cids"]), f"scores_{split}.npz")
    out = {"model": z["model"].astype(np.float32)[np.ix_(rq, rc)], "emb": None,
           "comps": {k[5:]: z[k].astype(np.float32)[np.ix_(rq, rc)] for k in z.files
                     if k.startswith("comp_") and k[5:] not in LEXICAL_NAMES}}
    ep = os.path.join(d, f"{split}_embeddings.npz")
    if os.path.exists(ep):
        try:
            e = np.load(ep, allow_pickle=False)
            eq, ec = _align(sp, _ids(e["qids"]), _ids(e["cids"]), f"{split}_embeddings.npz")
            out["emb"] = (e["C"].astype(np.float32)[ec], e["Q"].astype(np.float32)[eq])
        except Exception as ex:
            log(f"   (no usable embeddings for {split}: {ex})")
    return out


def run_name(d, taken):
    n = re.sub(r"^(cs781_phase7_|p7_)", "", os.path.basename(os.path.normpath(d))) or "run"
    base, i = n, 2
    while n in taken:
        n, i = f"{base}#{i}", i + 1
    return n


RUNS = {}
for d_ in discover_runs():
    if any(x and x in d_ for x in C.RUN_EXCLUDE):
        log(f"excluded {d_}")
        continue
    try:
        nm = run_name(d_, RUNS)
        RUNS[nm] = {"dir": d_, "dev": load_run(d_, SP["dev"], "dev"), "test": load_run(d_, SP["test"], "test")}
        log(f"loaded run '{nm}'  <- {d_}  (embeddings for PRF: dev {RUNS[nm]['dev']['emb'] is not None}, test {RUNS[nm]['test']['emb'] is not None})")
    except Exception as e:
        log(f"skipped {d_}: {e}")
def _what_is_visible():
    out = []
    for root in (INPUT_ROOT, BASE_DIR):
        if not os.path.isdir(root):
            out.append(f"  {root}: does not exist")
            continue
        out.append(f"  {root}: top level = {sorted(os.listdir(root))[:30]}")
        npz = []
        for d, _, files in os.walk(root, followlinks=True):
            npz += [os.path.join(d, f) for f in files if f.endswith(".npz")]
            if len(npz) > 40:
                break
        out.append(f"    .npz files found there: {npz[:40] if npz else 'NONE'}")
    return "\n".join(out)


if not RUNS:
    raise RuntimeError(
        "no usable run found: no folder under /kaggle/input or /kaggle/working holds BOTH scores_dev.npz and scores_test.npz "
        "(or the ones found were rejected - see 'skipped' lines above).\n"
        "Fix: add the output of your exp 2 / 3 / 4 run as an input (Add Input -> Notebook Output Files), or upload the four files "
        "scores_dev.npz, scores_test.npz, dev_embeddings.npz, test_embeddings.npz as a Kaggle Dataset and add that, or set RUN_DIRS "
        "in the config cell. Those files are written by the exp 2 / 3 / 4 notebooks of this series (cells 9 and 10) - NOT by the "
        "exp 1 notebook (phase7_model2_lora_docpool).\n\nWhat this notebook can see right now:\n" + _what_is_visible())
log(f"{len(RUNS)} run(s): {list(RUNS)}")


FUSION_SPACES = ("ensemble", "runs", "split")                 # simplest first


def run_components(split, space):
    """dense score components of every run, z-scored per query.  The more components, the more weights the dev set has to fit:
      ensemble : ONE component, the mean of the runs' saved model scores            (no dense weight to fit)
      runs     : one component per run (its saved model score)                      (one weight per run)
      split    : pooled / q2c / c2q of every run as separate components             (up to 3 weights per run)"""
    assert space in FUSION_SPACES, space
    out = {}
    for n, r in RUNS.items():
        d = r[split]
        if space == "split" and d["comps"]:
            out.update({f"{n}:{k}": zrows(v) for k, v in d["comps"].items()})
        else:
            out[n] = zrows(d["model"])
    if space == "ensemble":
        return {"ensemble": zrows(sum(out.values()) / len(out))}
    return out


def concat_embeddings(split):
    """one candidate-embedding matrix for PRF: the L2-normalised embeddings of every run side by side (cosine = mean of the runs')"""
    parts = [l2n(r[split]["emb"][0]) for r in RUNS.values() if r[split]["emb"] is not None]
    return l2n(np.concatenate(parts, axis=1)) if parts else None

### 7. Dev: every run alone, the fusion space, then the tuned recipe
The fusion space is chosen by held-out dev F1; weights, PRF, competition and refinement are then chosen on dev; each step must gain >= its MIN_GAIN.

In [ ]:
# =============================================================================
# CELL 7 - dev: every run on its own, a plain average, then the tuned recipe (fusion -> PRF -> competition -> refinement)
#
# Why the fusion space is chosen by cross-validation: the dev split is small (a hundred-odd queries), and a greedy search over
# 10-15 weights fits its noise (dev F1 0.43 -> grader 0.39 in an earlier run).  So three spaces are compared on HELD-OUT dev
# queries, repeated over several random splits, and the simplest one that is within SPACE_MARGIN of the best is used:
#     ensemble : the mean of the runs' scores + lexical / time components   (only the lexical weights are fitted)
#     runs     : one weight per run + lexical / time components
#     split    : pooled / q2c / c2q of every run separately + lexical / time (the largest space, the easiest to overfit)
# and the lexical / citation signals (BM25, short-query BM25, statute overlap, cite, cocite, cpop) use NO labels, so they are out-of-sample
# on the 800-odd TRAIN queries too: their relative weights can be fitted there ('train' lexical mode: one 'lex' component, only its
# weight is fitted on dev) instead of on 118 dev queries ('dev' mode).  Both are compared on held-out dev queries.
# =============================================================================
dev = SP["dev"]
dq, dc = [dev.qtext[q] for q in dev.qids], [dev.ctext[c] for c in dev.cids]


def on_weights(w):
    return {k: v for k, v in w.items() if v}


try:
    GS = [gold_structure(SP[s]) for s in ("train", "dev")]
    log("[pool structure] how many queries is a pool case gold for?")
    log(pd.DataFrame(GS).to_string(index=False))
except Exception as e:
    log(f"(pool-structure diagnostics skipped: {e!r})")

setup_lexical(dev)
try:
    lex_dev = lexical_components(dq, dc, dev.cids)
except Exception as e:
    log(f"!! lexical components failed ({e!r}); continuing without them")
    lex_dev = {}
Ce_dev = concat_embeddings("dev")

LEX_BLOCK = ("bm25", "bm25_top", "refs", "cite", "cocite", "cpop")          # label-free signals (time / tprior use train labels: not in the block)
LEX_W = {}
if C.LEX_MODE in ("auto", "train"):
    try:
        tr = SP["train"]
        lex_tr = {k: zrows(v) for k, v in lexical_components([tr.qtext[q] for q in tr.qids], [tr.ctext[c] for c in tr.cids]).items()
                  if k in LEX_BLOCK}
        if len(lex_tr) >= 2:
            prim_l = max(lex_tr, key=lambda n: f1_of(lex_tr[n], tr, C.K))
            w_l, f_l, f_l0 = tune_fusion(lex_tr, prim_l, tr, C.K, grid=GRID_V3, min_gain=C.MIN_GAIN)
            LEX_W = {k: w for k, w in w_l.items() if w}
            log(f"[lexical block] weights tuned on the TRAIN split ({len(tr.qids)} queries): {LEX_W} -> train F1@7 {f_l:.4f} "
                f"(best single signal '{prim_l}' {f_l0:.4f})")
        del lex_tr
    except Exception as e:
        LEX_W = {}
        log(f"(lexical block tuning on train skipped: {e!r})")


def assemble_components(Zd, lexZ, lex_mode):
    """dense components + lexical components.  lex_mode 'train': the label-free block is collapsed into ONE component 'lex' with the
    weights fitted on train; time / tprior stay separate.  lex_mode 'dev': every lexical component stays separate."""
    out = dict(Zd)
    if lex_mode == "train" and LEX_W:
        out["lex"] = zrows(fuse({k: lexZ[k] for k in LEX_W if k in lexZ}, LEX_W))
        out.update({k: v for k, v in lexZ.items() if k not in LEX_BLOCK})
    else:
        out.update(lexZ)
    return out


def space_components(space, lex_mode="dev"):
    Zd = run_components("dev", space)
    Z = assemble_components(Zd, {k: zrows(v) for k, v in lex_dev.items()}, lex_mode)
    prim = max(Zd, key=lambda n: f1_of(Zd[n], dev, C.K))
    return Zd, Z, prim


# ---- every component on its own (richest space) ----------------------------------------------------------------
rows = []
for k, Z in space_components("split", "dev")[1].items():
    mm = micro_prf(rank_topk(Z, dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
    rows.append({"component": k, "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
if len(RUNS) > 1:
    mm = micro_prf(rank_topk(space_components("ensemble", "dev")[0]["ensemble"], dev.qids, dev.cids, C.K), dev.gold, dev.qids, C.K)
    rows.append({"component": f"plain average of the {len(RUNS)} runs", "P@7": round(mm["P"], 4), "R@7": round(mm["R"], 4), "F1@7": round(mm["F1"], 4)})
log("[rescore] dev, every component on its own:")
log(pd.DataFrame(rows).to_string(index=False))


# ---- choose the fusion space on held-out dev queries -----------------------------------------------------------
def cv_mean(Z, prim):
    outs = [cv_estimate(Z, prim, dev, C.K, folds=C.CV_FOLDS, seed=C.SEED + 17 * s, min_gain=C.MIN_GAIN, use_compete=C.USE_COMPETE,
                        compete_min_gain=C.COMPETE_MIN_GAIN, Ce=Ce_dev, use_refine=C.USE_REFINE) for s in range(max(1, C.CV_REPEATS))]
    return {k: float(np.mean([o[k] for o in outs])) for k in outs[0]}


assert C.FUSION_SPACE == "auto" or C.FUSION_SPACE in FUSION_SPACES, f"FUSION_SPACE must be 'auto' or one of {FUSION_SPACES}"
assert C.LEX_MODE in ("auto", "dev", "train"), "LEX_MODE must be 'auto', 'dev' or 'train'"
# (dense space, lexical mode), simplest first
ORDER = [("ensemble", "train"), ("runs", "train"), ("ensemble", "dev"), ("runs", "dev"), ("split", "train"), ("split", "dev")]
CANDS = [(sp_, lm_) for sp_, lm_ in ORDER
         if (C.FUSION_SPACE in ("auto", sp_)) and (C.LEX_MODE in ("auto", lm_)) and (lm_ == "dev" or LEX_W)
         and not (sp_ == "ensemble" and len(RUNS) == 1)]
if not CANDS:                                                              # e.g. one run + FUSION_SPACE='ensemble'
    CANDS = [(C.FUSION_SPACE if C.FUSION_SPACE != "auto" else "runs", "dev")]
cname = lambda c: f"{c[0]} + lexical({c[1]})"
SPACE_CV = {}
if len(CANDS) > 1 and C.CV_FOLDS:
    for cand in CANDS:
        try:
            Zd_, Z_, prim_ = space_components(*cand)
            SPACE_CV[cand] = cv_mean(Z_, prim_)
            SPACE_CV[cand]["components"] = len(Z_)
        except Exception as e:
            log(f"(cross-validation of '{cname(cand)}' failed: {e!r})")
if SPACE_CV:
    metric = lambda d: d[[k for k in d if k != "components"][-1]]              # the most complete pipeline of that space
    best_score = max(metric(v) for v in SPACE_CV.values())
    CHOSEN = next(c for c in CANDS if c in SPACE_CV and metric(SPACE_CV[c]) >= best_score - C.SPACE_MARGIN)
    tab = pd.DataFrame([{"dense space + lexical weights fitted on": cname(c).replace("lexical(", "").rstrip(")"), "components": v["components"],
                         **{k: round(x, 4) for k, x in v.items() if k != "components"}} for c, v in SPACE_CV.items()])
    log(f"[fusion space] held-out dev F1@7, mean of {max(1, C.CV_REPEATS)} repeats x {C.CV_FOLDS} folds (tuning redone inside every fold):")
    log(tab.to_string(index=False))
    log(f">>> chosen: {cname(CHOSEN)} - the simplest one within {C.SPACE_MARGIN} F1 of the best held-out score ({best_score:.4f})")
else:
    CHOSEN = CANDS[0] if len(CANDS) == 1 or C.FUSION_SPACE != "auto" else (("runs", "train") if LEX_W and C.LEX_MODE != "dev" else ("runs", "dev"))
    log(f">>> chosen: {cname(CHOSEN)} (not compared by cross-validation)")
SPACE, LEXMODE = CHOSEN
SPACE_CV = {cname(c): v for c, v in SPACE_CV.items()}

Z_dense, Z_dev, PRIMARY = space_components(SPACE, LEXMODE)
log(f"primary component (best single one on dev): '{PRIMARY}'")
REC = None
for tag, kw in (("full recipe", {}), ("without competition", {"use_compete": False}),
                ("without competition and PRF", {"use_compete": False, "use_prf": False})):
    try:
        args = dict(use_prf=C.USE_PRF, use_compete=C.USE_COMPETE, min_gain=C.MIN_GAIN, compete_min_gain=C.COMPETE_MIN_GAIN,
                    use_refine=C.USE_REFINE)
        args.update(kw)
        REC, S_final_dev = tune_recipe(Z_dev, PRIMARY, Ce_dev, dev, C.K, **args)
        break
    except Exception as e:
        log(f"!! tuning '{tag}' failed ({e!r})")
assert REC is not None, "tuning failed in every configuration"
FD = REC["dev_f1"]
log(f"[recipe] dev F1@7: '{PRIMARY}' alone {FD['primary_alone']:.4f} -> weighted fusion {FD['fused']:.4f} -> "
    f"+PRF {FD['fused_prf']:.4f} -> +competition {FD.get('competition', FD['final']):.4f} -> +refinement {FD['final']:.4f}")
log("[recipe] " + describe_recipe(REC))
if C.USE_COMPETE:
    log("[recipe] note: competition is transductive (each candidate is compared across the other queries of the same split, "
        "no labels). Set USE_COMPETE=False for a per-query-only system.")
CV = SPACE_CV.get(cname(CHOSEN))
if CV is None and C.CV_FOLDS:
    try:
        CV = cv_mean(Z_dev, PRIMARY)
        CV["components"] = len(Z_dev)
    except Exception as e:
        log(f"(cross-validated estimate skipped: {e!r})")
if CV:
    log(f"[cv x{C.CV_FOLDS}] held-out dev F1@7 of the chosen space (tuning repeated inside each fold; PRF not included): "
        + " | ".join(f"{k} {v:.4f}" for k, v in CV.items() if k != "components"))
    log("[cv] the dev-tuned number below is optimistic: expect the grader to be closer to this held-out number than to the dev one.")
preds_dev = rank_topk(S_final_dev, dev.qids, dev.cids, C.K)
M_DEV = micro_prf(preds_dev, dev.gold, dev.qids, C.K)
log(f"\n>>> DEV  final rescored system: P@7={M_DEV['P']:.4f}  R@7={M_DEV['R']:.4f}  F1@7={M_DEV['F1']:.4f}")

### 8. Test predictions

In [ ]:
# =============================================================================
# CELL 8 - test predictions from the rescoring recipe.  Exactly 7 unique ids per test query, from the TEST pool only.
#   <OUT_DIR>/prediction.json                 final: runs + lexical/time components + PRF + competition   <- submit first
#   <OUT_DIR>/no_refinement/prediction.json   the same without the competition<->feedback refinement rounds (only if they are on)
#   <OUT_DIR>/no_competition/prediction.json  the same without the cross-query competition step         <- submit second
#   <OUT_DIR>/primary_only/prediction.json    the best single score component alone (reference)
# =============================================================================
def write_submission(pred, qids, pool, folder):
    os.makedirs(folder, exist_ok=True)
    pool = set(pool)
    for q in qids:
        p = pred[q]
        assert len(p) == C.K and len(set(p)) == C.K, f"query {q}: need {C.K} unique ids"
        assert all(c in pool for c in p), f"id outside the test pool for query {q}"
        assert q not in p, f"query {q} retrieved itself"
    path = os.path.join(folder, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump({q: pred[q] for q in qids}, f, indent=2)
    chk = json.load(open(path, encoding="utf-8"))
    assert list(chk.keys()) == list(qids) and all(len(v) == C.K and len(set(v)) == C.K for v in chk.values())
    with zipfile.ZipFile(os.path.join(folder, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    return path, len(chk)


te = SP["test"]
tq, tc = [te.qtext[q] for q in te.qids], [te.ctext[c] for c in te.cids]
Z_te = run_components("test", SPACE)
Z_primary_te = Z_te[PRIMARY]
write_submission(rank_topk(Z_primary_te, te.qids, te.cids, C.K), te.qids, te.cids, os.path.join(OUT_DIR, "primary_only"))   # a valid file first
try:
    Z_te = assemble_components(Z_te, {k: zrows(v) for k, v in lexical_components(tq, tc, te.cids).items()}, LEXMODE)
except Exception as e:
    log(f"!! lexical components failed on test ({e!r})")
assert set(Z_te) == set(REC["weights"]), f"components differ between dev {sorted(REC['weights'])} and test {sorted(Z_te)}"
Ce_te = concat_embeddings("test")
if (REC["prf_m"] is not None or REC.get("refine")) and Ce_te is None:
    raise RuntimeError("the recipe uses PRF / refinement but the runs have no test_embeddings.npz; re-run with USE_PRF=False and USE_REFINE=False")
S_final_te = apply_recipe(Z_te, Ce_te, te, REC)
if REC.get("refine"):
    S_nr = apply_recipe(Z_te, Ce_te, te, {**REC, "refine": None})
    p2, n2 = write_submission(rank_topk(S_nr, te.qids, te.cids, C.K), te.qids, te.cids, os.path.join(OUT_DIR, "no_refinement"))
    log(f"wrote {p2}  ({n2} queries)  <- same recipe with the competition but without the refinement rounds")
if REC["compete"] is not None:
    S_nc = apply_recipe(Z_te, Ce_te, te, {**REC, "compete": None, "refine": None})
    p1, n1 = write_submission(rank_topk(S_nc, te.qids, te.cids, C.K), te.qids, te.cids, os.path.join(OUT_DIR, "no_competition"))
    log(f"wrote {p1}  ({n1} queries)  <- same recipe without the cross-query competition step")
pred_te = rank_topk(S_final_te, te.qids, te.cids, C.K)
path, n = write_submission(pred_te, te.qids, te.cids, OUT_DIR)
shutil.copy(os.path.join(OUT_DIR, "submission.zip"), os.path.join(OUT_DIR, "submission_final.zip"))
log(f"wrote {path}  ({n} test queries x {C.K} ids) + submission.zip")
if C.MAX_QUERIES or C.MAX_POOL:
    log("WARNING: smoke run (MAX_QUERIES / MAX_POOL set): DO NOT SUBMIT.")
with open(os.path.join(OUT_DIR, "rescore_results.json"), "w", encoding="utf-8") as f:
    json.dump({"runs": {n: r["dir"] for n, r in RUNS.items()}, "fusion_space": SPACE, "lexical_mode": LEXMODE, "lexical_block_weights": LEX_W,
               "space_cv": SPACE_CV, "primary": PRIMARY,
               "recipe": REC, "cv_estimate": CV,
               "dev_final": M_DEV, "lexical_notes": LEX_NOTES}, f, indent=1,
              default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
log("example:", list(json.load(open(path)).items())[0])
log(f"artifacts in {OUT_DIR}: " + ", ".join(sorted(os.listdir(OUT_DIR))))